In [1]:

%pip -q install langgraph pandas matplotlib

In [2]:

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
import time

class AgentState(TypedDict):
    task: str
    category: str
    response: str

def classify_task(state: AgentState):
    task = state["task"].lower()
    if "refund" in task:
        category = "refund"
    elif "invoice" in task:
        category = "invoice"
    else:
        category = "general"
    return {"category": category}

def generate_response(state: AgentState):
    responses = {
        "refund": "Route to refund support.",
        "invoice": "Route to invoice support.",
        "general": "Route to general support."
    }
    return {"response": responses[state["category"]]}

builder = StateGraph(AgentState)
builder.add_node("classify", classify_task)
builder.add_node("respond", generate_response)
builder.add_edge(START, "classify")
builder.add_edge("classify", "respond")
builder.add_edge("respond", END)

agent = builder.compile()

result = agent.invoke({
    "task": "I need help with my invoice",
    "category": "",
    "response": ""
})

print(result)

{'task': 'I need help with my invoice', 'category': 'invoice', 'response': 'Route to invoice support.'}


In [3]:

import pandas as pd
import time

test_cases = [
    ("I need a refund", "refund"),
    ("Where is my invoice?", "invoice"),
    ("Hello, I need assistance", "general"),
    ("Please refund my payment", "refund"),
    ("Send me the invoice", "invoice"),
    ("How do I contact support?", "general"),
    ("My refund is delayed", "refund"),
    ("Invoice number is missing", "invoice"),
    ("I have a question", "general"),
    ("I want to cancel my order", "general")
]

records = []

for task, expected in test_cases:
    start = time.perf_counter()

    output = agent.invoke({
        "task": task,
        "category": "",
        "response": ""
    })

    latency = (time.perf_counter() - start) * 1000

    records.append({
        "task": task,
        "expected": expected,
        "predicted": output["category"],
        "correct": output["category"] == expected,
        "latency_ms": latency
    })

df = pd.DataFrame(records)

print(df.to_string(index=False))
print("\nAccuracy:", round(df["correct"].mean() * 100, 2), "%")
print("Average latency:", round(df["latency_ms"].mean(), 2), "ms")

df.to_csv("agenticops_baseline_results.csv", index=False)

                     task expected predicted  correct  latency_ms
          I need a refund   refund    refund     True   12.334412
     Where is my invoice?  invoice   invoice     True   11.911643
 Hello, I need assistance  general   general     True   10.421451
 Please refund my payment   refund    refund     True    6.660674
      Send me the invoice  invoice   invoice     True    5.285336
How do I contact support?  general   general     True    4.300579
     My refund is delayed   refund    refund     True    6.327131
Invoice number is missing  invoice   invoice     True    3.193162
        I have a question  general   general     True    8.952419
I want to cancel my order  general   general     True    6.331529

Accuracy: 100.0 %
Average latency: 7.57 ms


In [4]:

accuracy = df["correct"].mean()
failure_rate = 1 - accuracy
avg_latency = df["latency_ms"].mean()

accuracy_threshold = 0.90
failure_budget = 0.10

release_passed = (
    accuracy >= accuracy_threshold
    and failure_rate <= failure_budget
)

print("Accuracy:", round(accuracy * 100, 2), "%")
print("Failure rate:", round(failure_rate * 100, 2), "%")
print("Average latency:", round(avg_latency, 2), "ms")
print("Release decision:", "PASS" if release_passed else "FAIL")

Accuracy: 100.0 %
Failure rate: 0.0 %
Average latency: 7.57 ms
Release decision: PASS


In [5]:

from langgraph.graph import StateGraph, START, END
import pandas as pd

# Candidate agent with an intentional regression
def faulty_classify(state):
    task = state["task"].lower()

    if "refund" in task:
        category = "general"  # Deliberate defect
    elif "invoice" in task:
        category = "invoice"
    else:
        category = "general"

    return {"category": category}

candidate_builder = StateGraph(AgentState)
candidate_builder.add_node("classify", faulty_classify)
candidate_builder.add_node("respond", generate_response)
candidate_builder.add_edge(START, "classify")
candidate_builder.add_edge("classify", "respond")
candidate_builder.add_edge("respond", END)

candidate_agent = candidate_builder.compile()

# Compare baseline and candidate on identical tests
comparison = []

for task, expected in test_cases:
    initial = {
        "task": task,
        "category": "",
        "response": ""
    }

    baseline = agent.invoke(initial)
    candidate = candidate_agent.invoke(initial)

    comparison.append({
        "task": task,
        "expected": expected,
        "baseline": baseline["category"],
        "candidate": candidate["category"],
        "baseline_correct": baseline["category"] == expected,
        "candidate_correct": candidate["category"] == expected
    })

results = pd.DataFrame(comparison)

baseline_accuracy = results["baseline_correct"].mean()
candidate_accuracy = results["candidate_correct"].mean()

print("BASELINE ACCURACY:", f"{baseline_accuracy:.0%}")
print("CANDIDATE ACCURACY:", f"{candidate_accuracy:.0%}")
print("ACCURACY CHANGE:",
      f"{(candidate_accuracy - baseline_accuracy)*100:.1f} percentage points")

# Example policy: candidate must achieve at least 90%
# accuracy and must not degrade from the baseline.
release_approved = (
    candidate_accuracy >= 0.90
    and candidate_accuracy >= baseline_accuracy
)

print("RELEASE DECISION:",
      "APPROVED" if release_approved else "BLOCKED")

print("\nFailed candidate cases:")
print(results[~results["candidate_correct"]].to_string(index=False))

results.to_csv("agenticops_regression_comparison.csv", index=False)

BASELINE ACCURACY: 100%
CANDIDATE ACCURACY: 70%
ACCURACY CHANGE: -30.0 percentage points
RELEASE DECISION: BLOCKED

Failed candidate cases:
                    task expected baseline candidate  baseline_correct  candidate_correct
         I need a refund   refund   refund   general              True              False
Please refund my payment   refund   refund   general              True              False
    My refund is delayed   refund   refund   general              True              False


In [6]:

import json
import time

task = "I need help with my invoice"

print("AGENT EXECUTION TRACE")
print("=" * 45)

start = time.perf_counter()

for event in agent.stream(
    {
        "task": task,
        "category": "",
        "response": ""
    },
    stream_mode="updates"
):
    print(json.dumps(event, indent=2))

elapsed_ms = (time.perf_counter() - start) * 1000

print("=" * 45)
print(f"Total execution time: {elapsed_ms:.2f} ms")


AGENT EXECUTION TRACE
{
  "classify": {
    "category": "invoice"
  }
}
{
  "respond": {
    "response": "Route to invoice support."
  }
}
Total execution time: 9.46 ms


In [7]:

%pip -q install opentelemetry-api opentelemetry-sdk

In [8]:

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    SimpleSpanProcessor,
    ConsoleSpanExporter
)
from opentelemetry.trace import Status, StatusCode

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)

trace.set_tracer_provider(provider)
tracer = provider.get_tracer("agenticops.capstone")

def traced_agent_execution(task):
    with tracer.start_as_current_span("agent_execution") as span:
        span.set_attribute("agent.framework", "LangGraph")
        span.set_attribute("agent.task_category", "classification")

        try:
            state = {
                "task": task,
                "category": "",
                "response": ""
            }

            result = agent.invoke(state)

            span.set_attribute(
                "agent.predicted_category",
                result["category"]
            )
            span.set_attribute("agent.success", True)

            return result

        except Exception as error:
            span.record_exception(error)
            span.set_status(Status(StatusCode.ERROR))
            raise

result = traced_agent_execution(
    "I need help with my invoice"
)

print("\nAGENT RESULT:", result)

{
    "name": "agent_execution",
    "context": {
        "trace_id": "0xb9a3cd322f38ac8311c7f21fb0f0dd80",
        "span_id": "0x3d4473bc616322a5",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-10-24T02:24:47.148829Z",
    "end_time": "2026-10-24T02:24:47.151100Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "agent.framework": "LangGraph",
        "agent.task_category": "classification",
        "agent.predicted_category": "invoice",
        "agent.success": true
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.42.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}

AGENT RESULT: {'task': 'I need help with my invoice', 'category': 'invoice', 'response': 'Route to invo

In [10]:

from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

# Create an in-memory exporter
memory_exporter = InMemorySpanExporter()

provider_test = TracerProvider()
provider_test.add_span_processor(
    SimpleSpanProcessor(memory_exporter)
)

tracer_test = provider_test.get_tracer("agenticops.capstone")

# Execute agent with tracing
with tracer_test.start_as_current_span("agent_execution") as span:

    span.set_attribute("agent.framework", "LangGraph")

    result = agent.invoke({
        "task": "I need help with my invoice",
        "category": "",
        "response": ""
    })

    span.set_attribute("agent.category", result["category"])
    span.set_attribute("agent.success", True)

# Retrieve captured spans
spans = memory_exporter.get_finished_spans()

print("Total spans captured:", len(spans))

for s in spans:
    print("Span Name:", s.name)
    print("Trace ID:", format(s.context.trace_id, "032x"))
    print("Duration (ms):",
          round((s.end_time - s.start_time) / 1_000_000, 2))
    print("Attributes:", dict(s.attributes))

Total spans captured: 1
Span Name: agent_execution
Trace ID: 6e1c73cb2b2af28b53a5c62e14dbb66b
Duration (ms): 9.53
Attributes: {'agent.framework': 'LangGraph', 'agent.category': 'invoice', 'agent.success': True}


In [11]:

from langgraph.graph import StateGraph, START, END
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))
tracer = provider.get_tracer("agenticops.node_tracing")

def traced_classify(state):
    with tracer.start_as_current_span("classify_task") as span:
        result = classify_task(state)
        span.set_attribute("agent.category", result["category"])
        return result

def traced_respond(state):
    with tracer.start_as_current_span("generate_response") as span:
        result = generate_response(state)
        span.set_attribute("agent.response_generated", True)
        return result

builder = StateGraph(AgentState)
builder.add_node("classify", traced_classify)
builder.add_node("respond", traced_respond)
builder.add_edge(START, "classify")
builder.add_edge("classify", "respond")
builder.add_edge("respond", END)

traced_agent = builder.compile()

exporter.clear()

with tracer.start_as_current_span("agent_execution"):
    result = traced_agent.invoke({
        "task": "I need help with my invoice",
        "category": "",
        "response": ""
    })

print("Agent result:", result)
print("\nCAPTURED SPANS")

for span in exporter.get_finished_spans():
    print(
        span.name,
        "| Duration:",
        round((span.end_time - span.start_time) / 1_000_000, 2),
        "ms",
        "| Parent:",
        format(span.parent.span_id, "016x") if span.parent else "None"
    )

Agent result: {'task': 'I need help with my invoice', 'category': 'invoice', 'response': 'Route to invoice support.'}

CAPTURED SPANS
classify_task | Duration: 0.06 ms | Parent: 651c6fce4d9e23c3
generate_response | Duration: 0.02 ms | Parent: 651c6fce4d9e23c3
agent_execution | Duration: 2.41 ms | Parent: None
